<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day20_practice3_llm_service_%EC%99%84%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 감지된 "사진을 보고" 상황을 해석 + 심각도 판정 (멀티모달LLM)

In [15]:
import json, base64, os, time, random, requests, re

In [16]:
# API 키 - Colab Secrets에서 가져오기
def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get("VISION_API_KEY")
  except Exception:
    return os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

USE_REAL_AI = bool(_get_api_key()) # 키 있으면 True(실제 AI) 없으면 False(mock)
FALLBACK_ON_ERROR = True # real이 실패해도 mock으로 앱 실행

In [35]:
# 셀 4. _real_interpret : 사진을 "보는" LLM 멀티모달 (Vision)
def _encode_image(image_path):
  with open(image_path, "rb") as f:
    return base64.b64encode(f.read()).decode("utf-8") # 이미지 바이트를 텍스트(base64 문자열)로 포장, API로 사진을 보내는 표준 방법

# API 호출 - openAI 호환 api 형식으로 코드 작성
VISION_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"
VISION_MODELS = ["gemini-flash-lite-latest"] # Google이 업데이트하는 Flash-Lite 라인업의 가장 최신 버전으로 자동 연결

RETRY_STATUS = {429, 500, 502, 503, 504} # 잠깐 뒤 다시 시도하면 되는 상태 코드, 429 요청과다, 500번대는 서버문제
MAX_RETRIES = 4 # 모델당 최대 시도 횟수
TIMEOUT_SEC = 60

def _post_with_retry(payload, headers): # 재시도 함수 (요청 본문, 인증 정보)
  last_err = None
  for model in VISION_MODELS:
    payload["model"] = model
    for attempt in range(MAX_RETRIES):
      try:
        r = requests.post(f"{VISION_BASE_URL}/chat/completions", headers=headers, json=payload, timeout=TIMEOUT_SEC)
      except (requests.Timeout, requests.ConnectionError) as e:
        last_err = e
        r = None
      if r is not None:
        if r.status_code not in RETRY_STATUS:
          r.raise_for_status()
          return r
        last_err = requests.HTTPError(
            f"{r.status_code} (model={model}, 시도{attempt+1}/{MAX_RETRIES})", response=r
        )
        if attempt < MAX_RETRIES - 1:
          wait = 2 ** attempt + random.uniform(0, 0.5)
        if r is not None and r.headers.get("Retry-After", "").isdigit(): # .isdigit(): 문자열이 숫자로만 이뤄졌으면 True
          wait = min(int(r.headers["Retry-After"]), 20) # 20초로 제한
        print(f"[Vision] {model} 일시 오류 → {wait:.1f}초 뒤 재시도 ({attempt+1}/{MAX_RETRIES-1})")
        time.sleep(wait)
    print(f"[Vision] {model} 계속 실패 → 다음 모델 시도")
  raise last_err # 전부 실패 시 예외를 직접 발생시킴

def _real_interpret(image_path, detections): # 사진을 "보는" 멀티모달 해석
  api_key = _get_api_key()
  detected = ", ".join(str(d.get("type", "위험")) for d in detections) # 감지된 위험 종류들 "포트홀, 낙하물"
  prompt = f"""당신은 도로 안전 전문가입니다.
  이 사진에서 {detected}이(가) 감지되었습니다. 사진을 보고 아래 JSON 형식으로만 답하세요:
  {{
    'description': '상황을 2~3문장으로 설명 (심각도, 위치, 위험성)',
    'ai_severity': 1~5 사이 정수 (5가 가장 위험, 반드시 숫자로),
    'location_hint': '도로 종류/위치 추정',
    'risk_factors': ['구체적 위험 요인들']
  }}"""
  img_b64 = _encode_image(image_path) # 사진을 API로 보내려고 base64 텍스트로 변환
  payload = {
      "messages": [{
          "role": "user",
          "content": [ # 글 + 이미지
              {"type": "text",
               "text": prompt},
              {"type": "image_url",
               "image_url": {
                   "url": f"data:image/jpeg;base64,{img_b64}"}},
          ],
      }],
      "response_format": {"type": "json_object"}, # 모델이 JSON만 출력하도록 강제
      "max_tokens": 2000, # 낮을수록 밀관되고 안정적인 답 (분석.JSON 출력에 적합)
      "temperature": 0.2,
  }
  headers = {"Authorization": f"Bearer {api_key}"}
  r = _post_with_retry(payload, headers)
  content = r.json()["choices"][0]["message"]["content"]
  try:
    return json.loads(content)
  except (json.JSONDecodeError, TypeError):
    m = re.search(r"\{.*\}", content or "", re.DOTALL) # 첫 { 부터 마지막 } 까지
    if m:
      try:
        return json.loads(m.group())
      except json.JSONDecodeError:
        pass
    print(f"[Vision] JSON 파싱 실패 (응답 앞 200자): {(content or '')[:200]}")
    return {}

In [36]:
def _as_int(v, default=3, lo=1, hi=5):
  try:
    n = int(float(v))
  except (TypeError, ValueError):
    return default
  return max(lo, min(hi, n))

In [37]:
def _mock_interpret(image_path, detections):
    types = ", ".join(str(d.get("type", "위험")) for d in detections)
    max_sev = max((_as_int(d.get("severity"), 3) for d in detections), default=3)
    return {
        "description": f"전방 도로에 {types}이(가) 감지되었습니다. "
                       f"특히 심각도가 높은 위험이 있어 감속 및 우회가 권장됩니다.",
        "ai_severity": max_sev,
        "location_hint": "도심 도로",
        "risk_factors": ["차량 하부 손상 위험", "타이어 파손 가능", "우천 시 위험 증가"],
    }

In [38]:
def _normalize(result, detections):
  if not isinstance(result, dict):
    result = {}
  fallback_sev = max((_as_int(d.get("severity"), 3) for d in detections or []),default=3)
  raw = result.get("risk_factors")
  if isinstance(raw, list):
    factors = [str(x) for x in raw if x is not None]
  elif raw:
    factors = [str(raw)]
  else:
    factors = []
  return {
      "description": str(result.get("description") or "해석을 생성하지 못했습니다."),
      "ai_severity": _as_int(result.get("ai_severity"), default=fallback_sev),
      "location_hint": str(result.get("location_hint") or "위치 확인 중"),
      "risk_factors": factors,
  }

In [39]:
# 셀 5. 공개 함수 + 실행 검증
def interpret_situation(image_path, detections): # 서버가 부르는 함수
  if not USE_REAL_AI:
    return _normalize(_mock_interpret(image_path, detections), detections) # 가짜 데이터를 넣어서 얻은 결과를 검증 함수에 통과시킴
  try:
    return _normalize(_real_interpret(image_path, detections), detections) # 진짜AI에 데이터를 넣어서 얻은 결과를 검증 함수에 통과시킴
  except Exception as e:
    print(f"[LLM] 해석 실패: {type(e).__name__}: {e}")
    if not FALLBACK_ON_ERROR: # 폴백 끄면 지금 처리 중인 예외를 그대로 다시 던짐
      raise
    print("[LLM] 가짜 해석으로 대체합니다")
    return _normalize(_mock_interpret(image_path, detections), detections)

print(f"모드: {'REAL (키 감지)' if USE_REAL_AI else 'MOCK'}")
감지 = [{"type": "포트홀", "confidence": 0.92, "severity": 4,"box": {"x": 0.15, "y": 0.55, "w": 0.25, "h": 0.20}}]
IMAGE = "road_photo.png"
해석 = interpret_situation(IMAGE, 감지)
print("해석:", 해석["description"][:40], f".../심각도 {해석['ai_severity']}")

모드: REAL (키 감지)
해석: 도시 중심가 왕복 차선 중앙 부분에 심각한 균열과 포트홀이 발생해 있습니 .../심각도 4


In [40]:
print(해석)

{'description': '도시 중심가 왕복 차선 중앙 부분에 심각한 균열과 포트홀이 발생해 있습니다. 차량 통행이 많은 도로로, 주행 중 차량 타이어 파손 및 조향 상실 위험이 매우 큽니다. 즉각적인 보수 공사가 필요합니다.', 'ai_severity': 4, 'location_hint': '도심 시가지 왕복 4차선 이상 도로', 'risk_factors': ['차량 통행량 많음', '대형 크랙 및 함몰', '타이어 파손 위험', '운전자 조향 방해']}


In [41]:
assert set(해석.keys()) == {"description", "ai_severity", "location_hint", "risk_factors"}
assert isinstance(해석["ai_severity"], int) and 1 <= 해석["ai_severity"] <= 5
assert isinstance(해석["risk_factors"], list)
print("계약 검증 통과 (4키 + ai_severity는 int)")

계약 검증 통과 (4키 + ai_severity는 int)
